# S2 · Из измерений в таблицу

Дополнительный материал, 55 минут. Отдельное занятие S2 в текущем расписании не проводится и для S3/S4 не требуется.

Небольшой синтетический CSV содержит три измерения температуры на запуск. В нём специально есть пропуск, текст вместо числа, неизвестная метка и точный повтор. После проверки получим те же десять температур и меток, что в S1. Чтение файла подготовлено; вычисления объясняются последовательно.

### Подготовлено: чтение CSV и папка результата

In [1]:
import csv
from pathlib import Path

for folder in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / "starter"]:
    if (folder / "data" / "intro-temperature.csv").is_file():
        starter_folder = folder
        break
else:
    raise FileNotFoundError("Откройте notebook из starter или запустите код из папки курса")

with (starter_folder / "data" / "intro-temperature.csv").open(encoding="utf-8") as file:
    raw_rows = list(csv.DictReader(file))


## 1. Прочитать одну строку

CSV хранит текст. DictReader связывает заголовок столбца со значением строки. row['t1_c'] возвращает строку '28'. Всего исходных строк 14.

In [2]:
first_row = raw_rows[0]
print(first_row)
print("Номер запуска:", first_row["run_id"])
print("Первое измерение:", first_row["t1_c"])
print("Тип значения:", type(first_row["t1_c"]))
print("Строк в файле:", len(raw_rows))


{'run_id': 'F01', 't1_c': '28', 't2_c': '30', 't3_c': '32', 'inspection': '0'}
Номер запуска: F01
Первое измерение: 28
Тип значения: <class 'str'>
Строк в файле: 14


## 2. Проверить метки и измерения

Сначала проверяем метку и пустые поля. float переводит текст в число, int — метку в целое число. try/except позволяет записать причину ошибки преобразования. continue переходит к следующей строке. Остаётся 11 строк; три причины видны в rejected.

In [3]:
valid_rows = []
rejected = []

for row in raw_rows:
    run_id = row["run_id"]
    if row["inspection"] not in ["0", "1"]:
        rejected.append([run_id, "неизвестная метка"])
        continue

    if row["t1_c"] == "" or row["t2_c"] == "" or row["t3_c"] == "":
        rejected.append([run_id, "пропущено измерение"])
        continue

    try:
        t1 = float(row["t1_c"])
        t2 = float(row["t2_c"])
        t3 = float(row["t3_c"])
    except ValueError:
        rejected.append([run_id, "температура не является числом"])
        continue

    target = int(row["inspection"])
    valid_rows.append([run_id, t1, t2, t3, target])

print("Отклонённые строки:", rejected)
print("Строк после проверки:", len(valid_rows))


Отклонённые строки: [['F11', 'пропущено измерение'], ['F12', 'температура не является числом'], ['F13', 'неизвестная метка']]
Строк после проверки: 11


## 3. Удалить точный повтор

Сравнивается вся строка. Полностью одинаковая запись F03 встречается второй раз. Удаляется один точный повтор, остаются 10 строк. Совпадение только номера запуска при разных значениях потребовало бы отдельного разбора.

In [4]:
unique_rows = []
duplicates = 0

for row in valid_rows:
    if row in unique_rows:
        duplicates = duplicates + 1
    else:
        unique_rows.append(row)

print("Точных повторов:", duplicates)
print("Уникальных строк:", len(unique_rows))


Точных повторов: 1
Уникальных строк: 10


## 4. Из трёх измерений получить один признак

Среднее — сумма трёх измерений, делённая на три. append добавляет значение в список. Метка переносится из той же строки. Получатся данные обучающей части S1.

In [5]:
run_ids = []
temperature = []
target = []

for row in unique_rows:
    run_id = row[0]
    mean_temperature = (row[1] + row[2] + row[3]) / 3
    inspection = row[4]

    run_ids.append(run_id)
    temperature.append(mean_temperature)
    target.append(inspection)

print("Температуры:", temperature)
print("Метки:", target)


Температуры: [30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0]
Метки: [0, 0, 0, 1, 0, 0, 1, 1, 1, 1]


## 5. Проверить связь признака с исходной строкой

Три списка сохраняют общий порядок. Срез [1:4] берёт элементы с номерами 1, 2, 3. Для F01: (28+30+32)/3=30. Исходные измерения остаются в unique_rows.

In [6]:
for i in range(len(run_ids)):
    print(run_ids[i], "средняя температура:", temperature[i], "метка:", target[i])

first_measurements = unique_rows[0][1:4]
print("F01, исходные измерения:", first_measurements)
print("F01, среднее:", sum(first_measurements) / len(first_measurements))


F01 средняя температура: 30.0 метка: 0
F02 средняя температура: 35.0 метка: 0
F03 средняя температура: 40.0 метка: 0
F04 средняя температура: 45.0 метка: 1
F05 средняя температура: 50.0 метка: 0
F06 средняя температура: 55.0 метка: 0
F07 средняя температура: 60.0 метка: 1
F08 средняя температура: 65.0 метка: 1
F09 средняя температура: 70.0 метка: 1
F10 средняя температура: 75.0 метка: 1
F01, исходные измерения: [28.0, 30.0, 32.0]
F01, среднее: 30.0


## 6. Сохранить понятную таблицу

Сохранение — подготовленная техническая операция. writerow пишет одну строку CSV. Файл reports/s2-temperature-features.csv содержит идентификатор, среднюю температуру в градусах Цельсия и метку. Повтор запуска перезаписывает этот учебный результат.

In [7]:
output_folder = starter_folder / "reports"
output_folder.mkdir(exist_ok=True)
output_path = output_folder / "s2-temperature-features.csv"

with output_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["run_id", "mean_temperature_c", "inspection"])
    for i in range(len(run_ids)):
        writer.writerow([run_ids[i], temperature[i], target[i]])

print("Сохранено: reports/s2-temperature-features.csv")


Сохранено: reports/s2-temperature-features.csv
